# Capítulo 3 — Ajustar un modelo · Notebook para completar

*Matemáticas para la IA, paso a paso*, Belk Siri — notebook complementario.

[Abrir en Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/es/ch03_a_completer.ipynb) · Página complementaria: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/es.html

Las celdas que debes modificar llevan la indicación **PARA COMPLETAR**. Ejecuta las celdas en orden: mientras falte una respuesta, las comprobaciones lo indican sin error. La solución está en `ch03_corrige.ipynb`, en la misma página complementaria.

**Pérdida utilizada**: el error cuadrático medio (ecuación 3.2) en las viviendas A a F, salvo que se indique lo contrario.

**Contenido.** Datos · Los cálculos de las lecciones 3.1 a 3.6 · Ejercicios 3.1 a 3.6

*La celda de la lección 3.4 y la del ejercicio 3.4 f) encadenan cientos de miles de pasos: cuenta unos segundos para cada una.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Colores del libro: un color = un papel (observado, predicho, desviación, recta del colega)
BLUE, ORANGE, RESIDUAL, GREEN = '#2a78d6', '#eb6834', '#c2185b', '#2e8b57'

## Datos

El identificador D0.1 es el del libro y del archivo CSV.

In [ ]:
# Datos del capítulo 3, copiados del archivo CSV del libro (no modificar)

# D0.1 — las seis viviendas de la introducción, usadas para el ajuste
homes = pd.DataFrame({
    'home':   ['A', 'B', 'C', 'D', 'E', 'F'],
    'floor_area': [32, 45, 54, 70, 85, 98],
    'price':  [118, 172, 185, 245, 279, 335],
})
x = homes['floor_area'].to_numpy(dtype=float)
y = homes['price'].to_numpy(dtype=float)

# D3.1 — cuatro ventas nuevas, usadas solo para evaluar (lección 3.6)
new_sales = pd.DataFrame({
    'sale':      ['V1', 'V2', 'V3', 'V4'],
    'floor_area': [38, 58, 76, 92],
    'price':  [132, 215, 254, 322],
})
xv = new_sales['floor_area'].to_numpy(dtype=float)
yv = new_sales['price'].to_numpy(dtype=float)

# Vivienda G (introducción): ejemplo de cálculo, nunca usada para ajustar
x_G, y_G = 62, 209

### Las variables de los datos

Los nombres del código están en inglés. Para las columnas, la última columna da el nombre utilizado en los archivos CSV del libro.

| Nombre | Significado | Unidad | Columna del CSV |
|:--|:--|:--|:--|
| `home` | identificador de la vivienda (A, B…) | — | `logement` |
| `floor_area` | superficie habitable de la vivienda | m² | `surface_m2` |
| `price` | precio de venta | k€ | `prix_kEUR` |
| `sale` | identificador de la venta nueva (V1 a V4) | — | `vente` |

## Los cálculos de las lecciones, en Python

Estas celdas rehacen los cálculos de las lecciones 3.1 a 3.6.

In [ ]:
# Lección 3.1 — dos rectas, dos medidas de error (ecuaciones 3.1 y 3.2)
def mae(a, b):
    return np.mean(np.abs(y - (a * x + b)))

def mse(a, b):
    return np.mean((y - (a * x + b)) ** 2)

for name, a, b in [("introducción", 3, 30), ('colega', 3.3, 14)]:
    print(f'{name:10s} EAM = {mae(a, b):.2f} k€   ECM = {mse(a, b):.2f} k€²   RECM = {np.sqrt(mse(a, b)):.2f} k€')

In [ ]:
# Lección 3.2 — la curva de pérdida, b fijado en 30 k€ (ecuación 3.3)
slopes = np.round(np.arange(2.5, 3.45, 0.1), 1)
print(pd.Series([round(mse(a, 30), 2) for a in slopes], index=slopes))
A = np.linspace(2.4, 3.6, 241)
plt.figure(figsize=(6, 3))
plt.plot(A, [mse(a, 30) for a in A], color=BLUE)
plt.scatter(slopes, [mse(a, 30) for a in slopes], color='black', zorder=3)
plt.xlabel('Pendiente a (k€ por m²)'); plt.ylabel('Pérdida L(a) (k€²)'); plt.grid(alpha=0.4)
plt.show()

In [ ]:
# Lección 3.3 — secantes, derivada (ecuación 3.4) y dos actualizaciones (ecuación 3.5)
def derivative_a(a, b=30):
    e = y - (a * x + b)
    return -2 / len(x) * np.sum(x * e)

for h in (0.1, 0.01, 0.001):
    print(f'secante h = {h}: {(mse(2.9 + h, 30) - mse(2.9, 30)) / h:.1f}')
print('derivada exacta en 2,9:', round(derivative_a(2.9), 1))

a, eta = 2.5, 0.00006
for stage in range(3):
    print(f'paso {stage}: a = {a}, L = {mse(a, 30):.2f}, L\'(a) = {derivative_a(a):.2f}')
    a = round(a - eta * derivative_a(a), 3)    # redondeado a la milésima, como en el libro

In [ ]:
# Lección 3.4 — el gradiente (ecuación 3.6) y un paso sobre los dos coeficientes (ecuación 3.7)
def gradient(a, b):
    e = y - (a * x + b)
    return np.array([-2 / len(x) * np.sum(x * e), -2 / len(x) * np.sum(e)])

g = gradient(2.5, 30)
print('gradiente en (2,5 ; 30):', g.round(2))
print('tras un paso:', (np.array([2.5, 30]) - 0.00006 * g).round(4))

a, b = 2.5, 30.0
for k in range(1, 100001):
    a, b = np.array([a, b]) - 0.00006 * gradient(a, b)
    if k in (10, 1000, 100000):
        print(f'tras {k} pasos: a = {a:.4f}, b = {b:.4f}, L = {mse(a, b):.2f}')
a_ls, b_ls = np.polyfit(x, y, 1)
print(f'fondo del valle: a = {a_ls:.3f}, b = {b_ls:.2f}, L = {mse(a_ls, b_ls):.2f}')
print(f'G, ejemplo de cálculo: {a_ls * x_G + b_ls:.0f} k€ predichos para {y_G} k€ observados')

In [ ]:
# Lección 3.5 — dos descensos desde las mismas predicciones (ecuación 3.8)
mu, sigma = x.mean(), x.std()          # std() divide entre n, como en el capítulo 1
z = (x - mu) / sigma
print(f'mu = {mu:.0f} m², sigma = {sigma:.3f} m², z = {z.round(2)}')

def descent_m2(eta, n_steps, a=2.5, b=30.0):
    for _ in range(n_steps):
        a, b = np.array([a, b]) - eta * gradient(a, b)
    return a, b

def descent_std(eta, n_steps, a=2.5, b=30.0):
    w, c = a * sigma, b + a * mu        # ecuación 2.7: mismas predicciones al inicio
    for _ in range(n_steps):
        e = y - (w * z + c)
        w, c = w + eta * 2 / len(x) * np.sum(z * e), c + eta * 2 / len(x) * np.sum(e)
    return w / sigma, c - w / sigma * mu  # ecuación 3.8: vuelta a (a ; b)

print('pasos    | m², η = 0,000 06           | estandarizadas, η = 0,3')
a, b = 2.5, 30.0
done = 0
for k in (0, 1, 5, 10, 1000, 100000):
    a, b = descent_m2(0.00006, k - done, a, b); done = k
    a_s, b_s = descent_std(0.3, min(k, 1000))
    print(f'{k:>8} | a = {a:.3f} b = {b:6.2f} L = {mse(a, b):8.2f} | '
          f'a = {a_s:.3f} b = {b_s:6.2f} L = {mse(a_s, b_s):8.2f}')

# Tasas límite: más allá, el descenso diverge (unos 0,000 22 en m², 1 estandarizado)
for eta in (0.000216, 0.000217):
    print(f'm², η = {eta}: pérdida tras 400 pasos = {mse(*descent_m2(eta, 400)):.4g} k€²')
for eta in (0.99, 1.01):
    print(f'estandarizadas, η = {eta}: pérdida tras 400 pasos = {mse(*descent_std(eta, 400)):.4g} k€²')

In [ ]:
# Lección 3.6 — recta de mínimos cuadrados (ecuación 3.9) y R² (ecuación 3.10)
cov = np.mean((x - x.mean()) * (y - y.mean()))
a_ls = cov / x.var()
b_ls = y.mean() - a_ls * x.mean()
print(f'cov = {cov:.1f} m²·k€, var = {x.var():.2f} (m²)², a = {a_ls:.3f} k€/m², b = {b_ls:.2f} k€')

def r2_lesson(observed, predicted, reference):
    """R² = 1 − ECM de la regla / ECM de la predicción constante « reference »."""
    return 1 - np.mean((observed - predicted) ** 2) / np.mean((observed - reference) ** 2)

y_mean = y.mean()                      # precio medio de A a F: la referencia del R² en A a F
print(f'A a F: ECM constante = {np.mean((y - y_mean) ** 2):.2f}, ECM recta = {mse(a_ls, b_ls):.2f}, '
      f'R² = {r2_lesson(y, a_ls * x + b_ls, y_mean):.3f}, r² = {np.corrcoef(x, y)[0, 1] ** 2:.3f}')

# Las tres rectas, la misma medida, en A a F y luego en V1 a V4 (lección 3.6)
lines = {'mínimos cuadrados': (a_ls, b_ls), 'introducción': (3, 30), 'colega': (3.3, 14)}
for name, (a, b) in lines.items():
    e_af, e_v = y - (a * x + b), yv - (a * xv + b)
    print(f'{name:16s} A a F: ECM = {np.mean(e_af ** 2):6.2f}  EAM = {np.mean(np.abs(e_af)):5.2f}   |   '
          f'V1 a V4: ECM = {np.mean(e_v ** 2):6.2f}  EAM = {np.mean(np.abs(e_v)):5.2f}   desviaciones {e_v.round(2)}')
yv_bis = np.array([132, 215, 254, 312.0])   # si V4 se hubiera vendido a 312 k€
for name in ('mínimos cuadrados', 'colega'):
    a, b = lines[name]
    print(f'V4 a 312 k€: {name} ECM = {np.mean((yv_bis - (a * xv + b)) ** 2):.2f}')
print(f'G, ejemplo de cálculo: {a_ls * x_G + b_ls:.2f} k€ predichos, desviación {y_G - (a_ls * x_G + b_ls):.2f} k€')

## Ejercicio 3.1 — Leer un gráfico: dos medidas, dos mínimos

*Datos D0.1. Nociones: EAM y ECM (lección 3.1), curva de pérdida (lección 3.2), mediana (lección 1.3).* La pendiente está fijada en 3 k€/m². Responde primero a las preguntas a) a e) leyendo la figura del libro.

### a) a e) Tus lecturas

*Tus respuestas:* …

### f) En el notebook

In [ ]:
# PARA COMPLETAR: ECM y EAM de la recta 3x + b en A a F, para un valor de b
def mse_b(b):
    return None   # PARA COMPLETAR

def mae_b(b):
    return None   # PARA COMPLETAR

In [ ]:
# Comprobación y luego gráfico de las dos curvas
if mse_b(30) is None or mae_b(30) is None:
    print('Completa mse_b y mae_b.')
else:
    print('mse_b y mae_b:', 'correcto' if np.isclose(mse_b(30), 344 / 6) and np.isclose(mae_b(30), 44 / 6) else 'revisar')
    B = np.linspace(15, 45, 601)
    fig, (h, low) = plt.subplots(2, 1, figsize=(6, 4.5), sharex=True)
    h.plot(B, [mse_b(b) for b in B], color=BLUE); h.set_ylabel('ECM (k€²)'); h.grid(alpha=0.4)
    low.plot(B, [mae_b(b) for b in B], color=ORANGE); low.set_ylabel('EAM (k€)'); low.grid(alpha=0.4)
    low.set_xlabel('Ordenada en el origen b (k€)')
    plt.show()
    print('esquinas de la curva del EAM (r_i ordenados):', np.sort(y - 3 * x))

In [ ]:
# PARA COMPLETAR: mediana (ecuación 1.4) y media de los r_i = y_i − 3 x_i
median_r = None
mean_r = None

In [ ]:
# Comprobación
for name, yours, truth in [('mediana', median_r, 29.5), ('media', mean_r, 182 / 6)]:
    print(f'{name} de los r:', 'aún sin completar' if yours is None else ('correcto' if np.isclose(yours, truth) else 'revisar'))
if median_r is not None and mae_b(30) is not None:
    print(f'EAM en la mediana: {mae_b(median_r):.2f} k€ (en la meseta); '
          f'ECM en la media: {mse_b(mean_r):.2f} k€² (mínimo)')

*Tu conclusión:* …

## Ejercicio 3.2 — Dos pasos de descenso de gradiente

*Datos D0.1. Nociones: error cuadrático medio (lección 3.1), derivada y actualización (lección 3.3).* La pendiente está fijada en 3 k€/m². Buscamos la ordenada en el origen $b$. Haz primero las preguntas a) a d) en papel.

### a) Preparar

In [ ]:
# PARA COMPLETAR: r_i = y_i − 3 x_i para las seis viviendas (array de numpy)
r = None

In [ ]:
# Comprobación
if r is None:
    print('Completa la celda anterior.')
else:
    print('r:', 'correcto' if np.allclose(r, [22, 37, 23, 35, 24, 41]) else 'revisar')

### b) y c) En papel

Anota aquí tus resultados: pérdidas en k€², valores de $b$ en k€.

In [ ]:
# PARA COMPLETAR: tus resultados calculados a mano
L_b0 = None       # pérdida en b0 = 20
derivative_b0 = None # L'(20)
b1 = None
L_b1 = None
b2 = None
L_b2 = None

In [ ]:
# Comprobación (cálculo independiente)
def _L(b): return np.mean((y - 3 * x - b) ** 2)
def _d(b): return -2 / 6 * np.sum(y - 3 * x - b)
_b1 = 20 - 0.3 * _d(20); _b2 = _b1 - 0.3 * _d(_b1)
expected_values = {'L_b0': (L_b0, _L(20)), 'derivative_b0': (derivative_b0, _d(20)), 'b1': (b1, _b1),
            'L_b1': (L_b1, _L(_b1)), 'b2': (b2, _b2), 'L_b2': (L_b2, _L(_b2))}
for name, (yours, truth) in expected_values.items():
    status = 'aún sin completar' if yours is None else ('correcto' if abs(yours - truth) < 0.01 else 'revisar')
    print(f'{name}: {status}')

### d) Interpretar

*Tu respuesta:* …

### e) En el notebook

In [ ]:
# PARA COMPLETAR
def loss(b):
    """ECM de la recta y = 3x + b en A a F (ecuación 3.2)."""
    return None   # PARA COMPLETAR

def derivative(b):
    """L'(b) = −(2/n) Σ e_i (ecuación 3.6)."""
    return None   # PARA COMPLETAR

<details><summary><b>Pista</b> (haz clic si te atascas)</summary>

Calcula primero el array de desviaciones `e = y - (3 * x + b)`. `np.mean(e ** 2)` da la pérdida y `np.sum(e)` la suma de las desviaciones.

</details>

In [ ]:
def descent(b0, eta, n_steps):
    """Devuelve la lista de los b y la lista de las pérdidas, del paso 0 al paso n_steps."""
    bs, losses = [b0], [loss(b0)]
    for _ in range(n_steps):
        bs.append(bs[-1] - eta * derivative(bs[-1]))   # ecuación 3.5
        losses.append(loss(bs[-1]))
    return bs, losses

if loss(20) is None or derivative(20) is None:
    print('Completa loss y derivative.')
else:
    bs, losses = descent(20, 0.3, 2)
    print('dos pasos:', np.round(bs, 3), np.round(losses, 2), '→',
          'correcto' if np.allclose(bs, [20, 26.2, 28.68]) else 'revisar')
    bs, losses = descent(20, 0.3, 20)
    plt.figure(figsize=(6, 2.8))
    plt.plot(losses, marker='o', color=BLUE)
    plt.xlabel('Paso'); plt.ylabel('Pérdida (k€²)'); plt.grid(alpha=0.4)
    plt.show()
    print('b tras 20 pasos:', round(bs[-1], 3), '  media de los r:', round(np.mean(y - 3 * x), 3))

### f) Predecir y luego comprobar: $\eta = 1{,}2$

*Tu predicción, antes de ejecutar:* …

In [ ]:
if loss(20) is not None and derivative(20) is not None:
    fig, axes = plt.subplots(1, 2, figsize=(8, 2.8))
    for ax, eta in zip(axes, (0.3, 1.2)):
        bs, losses = descent(20, eta, 6)
        ax.plot(bs, marker='o', color=ORANGE if eta > 1 else BLUE)
        ax.axhline(np.mean(y - 3 * x), color='black', linewidth=0.8, linestyle='--')
        ax.set_title(f'η = {eta}', fontsize=10); ax.set_xlabel('Paso'); ax.set_ylabel('b (k€)')
        print(f'η = {eta} :', np.round(bs, 2))
    plt.tight_layout(); plt.show()

### g) Ejemplo de cálculo con G

In [ ]:
# PARA COMPLETAR: precio predicho para G con la pendiente 3 y b2, luego la desviación con signo (k€)
price_G = None
gap_G = None

In [ ]:
# Comprobación
if price_G is None or gap_G is None:
    print('Completa la celda anterior.')
else:
    print('G:', 'correcto' if np.isclose(price_G, 214.68) and np.isclose(gap_G, -5.68) else 'revisar')

*Tu comentario sobre G:* …

## Ejercicio 3.3 — ¿Qué significa «99 %»?

*Datos D0.1 y D3.1. Nociones: coeficiente de determinación, ajuste y evaluación (lección 3.6).* Haz primero las preguntas a) a f) en papel.

### a) a f) En papel

*Tus respuestas:* …

### g) En el notebook

In [ ]:
# PARA COMPLETAR: ECM de unas predicciones, en las ventas que recibe (ecuación 3.2)
def sales_mse(observed, predicted):
    """observed y predicted: arrays de numpy de la misma longitud."""
    return None   # PARA COMPLETAR

In [ ]:
# Comprobación: la clasificación de la lección 3.6 y luego el R² en A a F (referencia: precio medio de A a F)
if sales_mse(y, y) is None:
    print('Completa la función sales_mse.')
else:
    a_ls = np.mean((x - x.mean()) * (y - y.mean())) / x.var()
    b_ls = y.mean() - a_ls * x.mean()
    expected_values = {'mínimos cuadrados': ((a_ls, b_ls), 45.36, 93.29), 'introducción': ((3, 30), 57.33, 134.25),
                'colega': ((3.3, 14), 65.11, 70.73)}
    for name, ((a, b), att_af, att_v) in expected_values.items():
        e1, e2 = sales_mse(y, a * x + b), sales_mse(yv, a * xv + b)
        print(f'{name:16s} A a F: {e1:6.2f}   V1 a V4: {e2:6.2f} →',
              'correcto' if abs(e1 - att_af) < 0.006 and abs(e2 - att_v) < 0.006 else 'revisar')
    r2_af = 1 - sales_mse(y, a_ls * x + b_ls) / sales_mse(y, np.full(6, y.mean()))
    print(f'R² en A a F = {r2_af:.4f}, r² = {np.corrcoef(x, y)[0, 1] ** 2:.4f} →',
          'correcto' if np.isclose(r2_af, np.corrcoef(x, y)[0, 1] ** 2) else 'revisar')

## Ejercicio 3.4 — Un descenso con superficies estandarizadas

*Datos D0.1. Nociones: gradiente (lección 3.4), estandarización y reconversión (lección 3.5), recta de mínimos cuadrados (lección 3.6).*

### a) En papel

*Tu cálculo:* …

### b) Media, desviación típica y puntuaciones estandarizadas

In [ ]:
# PARA COMPLETAR: media y desviación típica de las superficies (división entre n), luego las seis puntuaciones z
mu = None
sigma = None
z = None

In [ ]:
# Comprobación
if z is None:
    print('Completa la celda anterior.')
else:
    print('puntuaciones z:', 'correcto' if np.isclose(np.mean(z), 0) and np.isclose(np.mean(z ** 2), 1) and np.isclose(mu, 64) else 'revisar')

### c) El gradiente en la escritura estandarizada

In [ ]:
# PARA COMPLETAR: las dos derivadas parciales (lección 3.5), devueltas en un array [dL/dw, dL/dc]
def standardized_gradient(w, c):
    return None   # PARA COMPLETAR

In [ ]:
# Comprobación en el punto de partida (mismas predicciones que (2,5 ; 30))
if z is None or standardized_gradient(1, 1) is None:
    print('Completa las celdas b) y c).')
else:
    w0, c0 = 2.5 * sigma, 30 + 2.5 * mu
    g0 = standardized_gradient(w0, c0)
    print(f'inicio: w0 = {w0:.2f}, c0 = {c0:.0f}, gradient = {g0.round(2)} →',
          'correcto' if np.allclose(g0, [-29.683, -64.667], atol=0.001) else 'revisar')

### d) Treinta pasos y luego vuelta a las unidades de origen

In [ ]:
# PARA COMPLETAR: 30 actualizaciones con eta = 0,3 desde (w0 ; c0), luego a y b (ecuación 3.8)
a_std = None
b_std = None

In [ ]:
# Comprobación: ecuación 3.9 y np.polyfit
a_ls = np.mean((x - x.mean()) * (y - y.mean())) / x.var()
b_ls = y.mean() - a_ls * x.mean()
print(f'ecuación 3.9: a = {a_ls:.3f}, b = {b_ls:.2f}; np.polyfit: {np.round(np.polyfit(x, y, 1), 3)}')
if a_std is None:
    print('Completa la celda anterior.')
else:
    print('recta obtenida:', 'correcto' if np.allclose([a_std, b_std], [a_ls, b_ls], atol=1e-4) else 'revisar')

### e) y f) La pérdida de los dos descensos y luego el número de pasos

In [ ]:
# Los dos descensos de la lección 3.5, paso a paso (nada que completar)
def path_m2(n_steps, eta=0.00006):
    a, b, losses = 2.5, 30.0, []
    for _ in range(n_steps + 1):
        losses.append(np.mean((y - a * x - b) ** 2))
        e = y - a * x - b
        a, b = a + eta * 2 / 6 * np.sum(x * e), b + eta * 2 / 6 * np.sum(e)
    return losses

def path_std(n_steps, eta=0.3):
    zz = (x - x.mean()) / x.std()
    w, c, losses = 2.5 * x.std(), 30 + 2.5 * x.mean(), []
    for _ in range(n_steps + 1):
        losses.append(np.mean((y - w * zz - c) ** 2))
        e = y - w * zz - c
        w, c = w + eta * 2 / 6 * np.sum(zz * e), c + eta * 2 / 6 * np.sum(e)
    return losses

plt.figure(figsize=(6, 3))
plt.semilogy(path_m2(30), marker='o', color=ORANGE, label='superficies en m², η = 0,000 06')
plt.semilogy(path_std(30), marker='s', color=BLUE, label='superficies estandarizadas, η = 0,3')
plt.axhline(45.364, color='black', linestyle='--', linewidth=0.8)
plt.xlabel('Número de pasos'); plt.ylabel('Pérdida (k€²)'); plt.legend(); plt.grid(alpha=0.4); plt.show()

In [ ]:
# f) Número de pasos para quedar a menos de 0,1 k€² del mínimo (unos segundos)
L_min = np.mean((y - (a_ls * x + b_ls)) ** 2)

def n_steps(X, a, b, eta, limit=10 ** 6):
    for k in range(1, limit + 1):
        e = y - a * X - b
        a, b = a + eta * 2 / 6 * np.sum(X * e), b + eta * 2 / 6 * np.sum(e)
        if np.mean((y - a * X - b) ** 2) - L_min < 0.1:
            return k

losses_std = path_std(30)
print('superficies estandarizadas, η = 0,3:', next(k for k, v in enumerate(losses_std) if v - L_min < 0.1), 'pasos')
print('superficies en m², η = 0,000 06:    ', n_steps(x, 2.5, 30.0, 0.00006), 'pasos')

*Tu interpretación:* …

## Ejercicio 3.5 — Predecir y luego comprobar: superficies en decenas de m²

*Datos D0.1. Nociones: cambio de escala (lección 2.5), gradiente (lección 3.4), estandarización (lección 3.5).*

### a) a d) Tus predicciones, antes de ejecutar

*a):* …

*b):* …

*c):* …

*d):* …

### e) Comprobar

In [ ]:
# PARA COMPLETAR: las superficies de A a F en decenas de m²
x_tens = None

In [ ]:
# Comprobación y luego las medidas (nada más que completar)
if x_tens is None:
    print('Completa la celda anterior.')
else:
    print('x_tens:', 'correcto' if np.allclose(x_tens, [3.2, 4.5, 5.4, 7.0, 8.5, 9.8]) else 'revisar')
    print('a) recta de mínimos cuadrados (pendiente, ordenada):', np.round(np.polyfit(x_tens, y, 1), 2))
    e0 = y - (25 * x_tens + 30)
    print(f'b) cociente en el paso 0: {np.sum(x_tens * e0) / np.sum(e0):.2f} decenas de m²')
    for eta in (0.021, 0.022):
        a, b = 25.0, 30.0
        for _ in range(200):
            e = y - a * x_tens - b
            a, b = a + eta * 2 / 6 * np.sum(x_tens * e), b + eta * 2 / 6 * np.sum(e)
        print(f'c) η = {eta}: pérdida tras 200 pasos = {np.mean((y - a * x_tens - b) ** 2):.4g} k€²')
    L_min = np.mean((y - np.polyval(np.polyfit(x, y, 1), x)) ** 2)
    a, b = 25.0, 30.0
    for k in range(1, 10 ** 6):
        e = y - a * x_tens - b
        a, b = a + 0.006 * 2 / 6 * np.sum(x_tens * e), b + 0.006 * 2 / 6 * np.sum(e)
        if np.mean((y - a * x_tens - b) ** 2) - L_min < 0.1:
            break
    print(f'd) η = 0,006: {k} pasos para quedar a menos de 0,1 k€² del mínimo')

### f) Interpretar

*Tu respuesta:* …

## Ejercicio 3.6 — Desafío: círculos en el mapa

*Datos D0.1. Nociones: estandarización (lecciones 1.4 y 3.5), derivadas parciales y gradiente (lección 3.4).* Las preguntas a) a f) se hacen en papel. El notebook comprueba tus resultados.

### a) a f) En papel

*Tus demostraciones:* …

### Comprobar en el notebook

In [ ]:
zz = (x - x.mean()) / x.std()
print(f'a) suma de los z = {np.sum(zz):.1e}, suma de los z² = {np.sum(zz ** 2):.4f}')

In [ ]:
# PARA COMPLETAR: m = (1/n) Σ z_i y_i, luego un paso de descenso con tasa eta desde (w ; c)
m = None

def step(w, c, eta):
    """Devuelve (w, c) tras una actualización (ecuación 3.7 con z en lugar de x)."""
    return None   # PARA COMPLETAR

In [ ]:
# Comprobación de b) y d)
if m is None or step(1, 1, 0.1) is None:
    print('Completa la celda anterior.')
else:
    print(f'b) m = {m:.2f} →', 'correcto' if np.isclose(m, np.mean((x - x.mean()) * (y - y.mean())) / x.std()) else 'revisar')
    w0, c0 = 2.5 * x.std(), 30 + 2.5 * x.mean()
    w1, c1 = step(w0, c0, 0.5)
    print(f'd) un paso con η = 0,5: (w ; c) = ({w1:.2f} ; {c1:.2f}) →',
          'correcto' if np.allclose([w1, c1], [m, y.mean()]) else 'revisar')
    for eta in (1.0, 1.05):
        w, c = w0, c0
        sequence = []
        for _ in range(4):
            w, c = step(w, c, eta)
            sequence.append(round(c, 2))
        print(f'   η = {eta}: c tras 1, 2, 3, 4 pasos = {sequence}')
    W, Cc = np.meshgrid(np.linspace(40, 100, 200), np.linspace(180, 260, 200))
    Z = np.mean([(yi - W * zi - Cc) ** 2 for zi, yi in zip(zz, y)], axis=0)
    plt.figure(figsize=(4.5, 4.5))
    plt.contour(W, Cc, Z, levels=[50, 100, 200, 400, 800, 1600], colors=BLUE)
    plt.gca().set_aspect('equal')
    plt.xlabel('w (k€ por desviación típica)'); plt.ylabel('c (k€)'); plt.title('e) Círculos'); plt.show()